In [18]:
import pandas as pd

In [19]:
# Reads in the two human-labeled datasets
df_1 = pd.read_csv('dataset_1_components/bluesky_posts_tagged.csv', dtype=str)
df_2 = pd.read_csv('dataset_1_components/human_categorized_data.csv', dtype=str)

In [20]:
# Removes all non-labeled posts from the bluesky dataset
x = df_1.dropna(subset='Educator')
y = df_1.dropna(subset='Student')
z = df_1.dropna(subset='Expert/keynote/researchers')

x.rename(columns={'Educator':'Label'}, inplace=True)
y.rename(columns={'Student':'Label'}, inplace=True)
z.rename(columns={'Expert/keynote/researchers':'Label'}, inplace=True)

# Drops all columns that are not needed for the final dataset
x.drop(columns=['uri','author_name','author_did','reply','link','Student','Expert/keynote/researchers'], inplace=True)
y.drop(columns=['uri','author_name','author_did','reply','link','Educator','Expert/keynote/researchers'], inplace=True)
z.drop(columns=['uri','author_name','author_did','reply','link','Educator','Student'], inplace=True)

# Order is important here for drop_duplicate function
ds_1 = pd.concat([z, y, x])

# Final labeled bluesky dataset
ds_1.drop_duplicates(subset='text',inplace=True)
ds_1.dropna(subset='text', inplace=True)

C:\Users\allin\AppData\Local\Temp\ipykernel_27180\2326842287.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x.rename(columns={'Educator':'Label'}, inplace=True)
C:\Users\allin\AppData\Local\Temp\ipykernel_27180\2326842287.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y.rename(columns={'Student':'Label'}, inplace=True)
C:\Users\allin\AppData\Local\Temp\ipykernel_27180\2326842287.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-co

In [21]:
# Veryify that no values were fropped (CONFIRMED)
print(x['Label'].value_counts())
print(y['Label'].value_counts())
print(z['Label'].value_counts())
print(ds_1['Label'].value_counts()) 

Label
0    1501
1     512
Name: count, dtype: int64
Label
1    22
Name: count, dtype: int64
Label
1    73
Name: count, dtype: int64
Label
0    1394
1     607
Name: count, dtype: int64


In [22]:
# Reads the papers dataset (Abishethvarman et al. 2023)
ds_2 = pd.read_csv('dataset_1_components/categorized_data.csv', dtype=str)
ds_2

,Unnamed: 0,tweet_id,original_text,text,sentiment,tasks,users,technologies,organizations,competencies,job_profiles,category
0,0,1.59783e+18,"@andrew___baker when i tried it, gpt-3 nailed ...",username when i tried it gpt3 nailed a number ...,neutral,nailed a number,person,NaN,GPT3,mathematics,NaN,NaN
1,1,1.59783e+18,@andrew___baker i just left academia (former a...,username i just left academia former assistant...,positive,NaN,NaN,NaN,GPT3,economics,"assistant lecturer, building cleaner",Educator
2,2,1.59783e+18,[gpt-3] this post on lesswrong discusses the p...,gpt3 this post on lesswrong discusses the pote...,neutral,NaN,academic,reduce,"GPT3, LessWrong",fraud detection,writer,Expert/keynote/researchers
3,3,1.59782e+18,2: for your second step in your curriculum weâ...,2 for your second step in your curriculum were...,positive,NaN,NaN,NaN,"RobertHaisfield, GPT3",work independently,NaN,NaN
4,4,1.59782e+18,<u+2728>openai gpt-3.5 series of models <u+272...,openai gpt35 series of models if youre a rese...,neutral,NaN,researcher,api,OAI API,NaN,computer scientist,Expert/keynote/researchers
...,...,...,...,...,...,...,...,...,...,...,...,...
608532,236270,1.65789e+18,<u+0001f913><u+0001f913><u+0001f913> thanks go...,thanks google bert ive saved a lot of time do...,positive,works fine bitcoin altcoin defi nft sorry chatgpt,source code,"bitcoin, google",NaN,"Source (digital game creation systems), search...",NaN,NaN
608533,236271,1.65789e+18,the benefits of going through this course are ...,the benefits of going through this course are ...,positive,NaN,NaN,NaN,"Participants, AI Data Governance Control Frame...",statistics,NaN,NaN
608534,236272,1.6579e+18,all-time college basketball starting 5s accord...,alltime college basketball starting 5s accordi...,neutral,NaN,NaN,NaN,Alltime College Basketball,NaN,NaN,NaN
608535,236273,1.6579e+18,i broke down the technical advancements as nlp...,i broke down the technical advancements as nlp...,neutral,NaN,beginner,"model, deep learning","NLP, GPT, AI","terminology, natural language processing, deep...",NaN,NaN


In [23]:
# Drops posts w/o jobs, correctly labels target groups
education = ['lecturer', 'teacher', 'tutor', 'university teaching assistant', 'librarian']
ds_2.dropna(subset='job_profiles', inplace=True)

labels = []
for i in ds_2['job_profiles']:
    contained = False
    for j in education:
        if j in i:
            contained = True
    if contained:
        labels.append(1)
    else:
        labels.append(0)

In [24]:
# Adds label, drops unnecessary columns
ds_2.loc[:, 'label'] = labels
ds_2.drop(columns=['Unnamed: 0', 'tweet_id','text','sentiment','tasks','users','technologies','organizations','competencies','job_profiles', 'category'], inplace=True)
ds_2.rename(columns={'label':'Label', 'original_text':'text'}, inplace=True)
ds_2

,text,Label
1,@andrew___baker i just left academia (former a...,1
2,[gpt-3] this post on lesswrong discusses the p...,0
4,<u+2728>openai gpt-3.5 series of models <u+272...,0
15,we further exam the quality of the generated s...,0
19,"for decades, many people in academia have trea...",0
...,...,...
608496,@forrrestjr @kirawontmiss some dude did this t...,1
608497,"btw, there are various pieces of news how gpt ...",0
608500,"rhett “mankind,” a digital artist based in aus...",0
608506,my professor is saying she has no issue with u...,1


In [25]:
# Drops repeated text and NaN values
ds_2.dropna(subset='text',inplace=True)
ds_2.drop_duplicates(subset='text', inplace=True)
print(ds_2['Label'].value_counts())

Label
0    25365
1    15216
Name: count, dtype: int64


In [26]:
# Combines the two datasets and saves
final_df = pd.concat([ds_1, ds_2], ignore_index=True)
final_df['Label'] = final_df['Label'].astype(int)
print(final_df['Label'].value_counts())

final_df.to_csv('final_script/dataset_1_cleaned.csv')

Label
0    26759
1    15823
Name: count, dtype: int64
